In [1]:
import os
import shutil
import numpy as np
import pandas as pd
import mdtraj as md


input_root = "palantir_data"         
output_root = "palantir_data_equal_frames"

proteins = ["3UTQ", "5C0F", "5N1Y", "pep_free"]
replicas = [0, 1, 2]

os.makedirs(output_root, exist_ok=True)

#
frame_rows = []

for protein in proteins:
    pdb = os.path.join(input_root, protein, "backbone.pdb")

    for rep in replicas:
        xtc = os.path.join(input_root, protein, f"{protein}_rep_{rep}_backbone.xtc")
        traj = md.load_xtc(xtc, top=pdb)

        frame_rows.append({
            "pdb_code": protein,
            "replica": rep,
            "xtc_path": xtc,
            "pdb_file": pdb,
            "n_frames": traj.n_frames,
            "n_atoms": traj.n_atoms
        })

df_frames = pd.DataFrame(frame_rows)

target_frames = int(df_frames["n_frames"].min())

print("[INFO] Frame counts:")
display(df_frames[["pdb_code", "replica", "n_frames", "n_atoms"]])

print(f"[INFO] Target frame count = {target_frames}")


def make_stride_indices(n_frames, target_frames):
    if n_frames < target_frames:
        raise ValueError(f"n_frames={n_frames} < target_frames={target_frames}")

    indices = np.linspace(0, n_frames - 1, target_frames)
    indices = np.round(indices).astype(int)

    # Ensure unique indices
    if len(np.unique(indices)) != target_frames:
        indices = np.floor(np.linspace(0, n_frames, target_frames, endpoint=False)).astype(int)

    if len(np.unique(indices)) != target_frames:
        raise ValueError("Could not create unique frame indices.")

    return indices


rows = []

for _, row in df_frames.iterrows():
    protein = row["pdb_code"]
    rep = int(row["replica"])
    in_xtc = row["xtc_path"]
    in_pdb = row["pdb_file"]
    n_frames = int(row["n_frames"])

    out_protein_dir = os.path.join(output_root, protein)
    os.makedirs(out_protein_dir, exist_ok=True)

    out_pdb = os.path.join(out_protein_dir, "backbone.pdb")
    out_xtc = os.path.join(out_protein_dir, f"{protein}_rep_{rep}_backbone.xtc")

    if not os.path.exists(out_pdb):
        shutil.copy(in_pdb, out_pdb)

    print(f"[INFO] Processing {protein} rep {rep}: {n_frames} -> {target_frames}")

    traj = md.load_xtc(in_xtc, top=in_pdb)

    selected_indices = make_stride_indices(n_frames, target_frames)
    traj_equal = traj[selected_indices]

    traj_equal.save_xtc(out_xtc)

    rows.append({
        "pdb_code": protein,
        "replica": rep,
        "input_xtc": in_xtc,
        "output_xtc": out_xtc,
        "input_pdb": in_pdb,
        "output_pdb": out_pdb,
        "original_frames": n_frames,
        "target_frames": target_frames,
        "selected_first_index": int(selected_indices[0]),
        "selected_last_index": int(selected_indices[-1]),
        "selected_indices": ",".join(map(str, selected_indices))
    })

    print(f"[INFO] Saved: {out_xtc} | frames={traj_equal.n_frames} | atoms={traj_equal.n_atoms}")

df_equalized = pd.DataFrame(rows)
metadata_path = os.path.join(output_root, "metadata_equal_minframes.csv")
df_equalized.to_csv(metadata_path, index=False)

print("\n[INFO] Equalized full-atom dataset created.")
print(f"[INFO] Metadata saved: {metadata_path}")

[INFO] Frame counts:


,pdb_code,replica,n_frames,n_atoms
0,3UTQ,0,876,1544
1,3UTQ,1,876,1544
2,3UTQ,2,876,1544
3,5C0F,0,1001,1548
4,5C0F,1,1001,1548
5,5C0F,2,1001,1548
6,5N1Y,0,875,1548
7,5N1Y,1,875,1548
8,5N1Y,2,875,1548
9,pep_free,0,5001,1504


[INFO] Target frame count = 875
[INFO] Processing 3UTQ rep 0: 876 -> 875
[INFO] Saved: palantir_data_equal_frames/3UTQ/3UTQ_rep_0_backbone.xtc | frames=875 | atoms=1544
[INFO] Processing 3UTQ rep 1: 876 -> 875
[INFO] Saved: palantir_data_equal_frames/3UTQ/3UTQ_rep_1_backbone.xtc | frames=875 | atoms=1544
[INFO] Processing 3UTQ rep 2: 876 -> 875
[INFO] Saved: palantir_data_equal_frames/3UTQ/3UTQ_rep_2_backbone.xtc | frames=875 | atoms=1544
[INFO] Processing 5C0F rep 0: 1001 -> 875
[INFO] Saved: palantir_data_equal_frames/5C0F/5C0F_rep_0_backbone.xtc | frames=875 | atoms=1548
[INFO] Processing 5C0F rep 1: 1001 -> 875
[INFO] Saved: palantir_data_equal_frames/5C0F/5C0F_rep_1_backbone.xtc | frames=875 | atoms=1548
[INFO] Processing 5C0F rep 2: 1001 -> 875
[INFO] Saved: palantir_data_equal_frames/5C0F/5C0F_rep_2_backbone.xtc | frames=875 | atoms=1548
[INFO] Processing 5N1Y rep 0: 875 -> 875
[INFO] Saved: palantir_data_equal_frames/5N1Y/5N1Y_rep_0_backbone.xtc | frames=875 | atoms=1548
[INFO]